## Bienvenidos al Laboratorio 3 para la Semana 1, Día 4

¡Hoy vamos a construir algo con valor inmediato!

En la carpeta `me` he puesto un solo archivo `linkedin.pdf` - es una descarga en PDF de mi perfil de LinkedIn.

¡Por favor, reemplázalo con el tuyo!

También he creado un archivo llamado `summary.txt`.

No vamos a usar herramientas todavía, las vamos a agregar mañana.


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Buscando paquetes</h2>
<span style="color:#00bfff;">En este laboratorio, usaremos el excelente paquete Gradio para crear interfaces de usuario rápidas, y también usaremos el popular lector de PDF PyPDF2. Puedes obtener guías para estos paquetes preguntando a ChatGPT o a Claude, y encontrarás todos los paquetes de código abierto en el repositorio <a href="https://pypi.org">https://pypi.org</a>.
</span>
        </td>
    </tr>
</table>

In [1]:
# Si no sabe para qué sirve alguno de estos paquetes, ¡siempre puede pedirle una guía a ChatGPT!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedinbea.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contactar
628741267 (Home)
solana93@gmail.com
www.linkedin.com/in/beatriz-solana
(LinkedIn)
Aptitudes principales
Scikit-learn
Seaborn
Linux
Languages
Inglés (Full Professional)
Italiano (Elementary)
Certifications
Seminarios - Saca partido a tus
estudios con las herramientas de IA
de Google
Domina la IA con Gemini
SQL 
HTML5, CSS3 y JavaScript. Nivel 1
Beatriz Solana Ros
Técnica en IA, Analítica y Formación Cooke ESP - Data Science
Mentor 4geeks Academy - Estudiante Dam /Junior data scientist /
Machine Learning | Python | Git | Machine Learning | Pandas |
PowerBI (...)
Cartagena y alrededores
Extracto
Apasionada por los datos y la tecnología.Con casi 10 años de
experiencia en el sector turismo y golf, he trabajado en ventas,
administración y gestión de reservas, desarrollando habilidades
en organización, atención al cliente y resolución de problemas.Mi
transición al análisis de datos y la inteligencia artificial ha sido un
paso natural en mi camino profesional, impulsado por mi pa

In [5]:
with open("me/sobremi.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Beatriz Mezcolanza"

In [8]:
system_prompt = f"Estás actuando como {name}. Estás respondiendo preguntas en el sitio web de {name}, en particular preguntas relacionadas con la carrera, la trayectoria, las habilidades y la experiencia de {name}. \
Tu responsabilidad es representar a {name} en las interacciones en el sitio web con la mayor fidelidad posible. \
Se te proporciona un resumen de la trayectoria y el perfil de LinkedIn de {name} que puedes usar para responder preguntas. \
Sé profesional y atractivo, como si hablaras con un cliente potencial o un futuro empleador que haya visitado el sitio web. \
Si no sabes la respuesta, dilo. Sé concisa en tus respuestas y no te enrolles, con gracia murciana (de Murcia, España) y maja"

system_prompt += f"\n\n## Resumen:\n{summary}\n\n## Perfil de LinkedIn:\n{linkedin}\n\n"
system_prompt += f"En este contexto, charla con el usuario, utilizando siempre el personaje de {name}."


In [9]:
system_prompt

'Estás actuando como Beatriz Mezcolanza. Estás respondiendo preguntas en el sitio web de Beatriz Mezcolanza, en particular preguntas relacionadas con la carrera, la trayectoria, las habilidades y la experiencia de Beatriz Mezcolanza. Tu responsabilidad es representar a Beatriz Mezcolanza en las interacciones en el sitio web con la mayor fidelidad posible. Se te proporciona un resumen de la trayectoria y el perfil de LinkedIn de Beatriz Mezcolanza que puedes usar para responder preguntas. Sé profesional y atractivo, como si hablaras con un cliente potencial o un futuro empleador que haya visitado el sitio web. Si no sabes la respuesta, dilo. Sé concisa en tus respuestas y no te enrolles, con gracia murciana (de Murcia, España) y maja\n\n## Resumen:\nSoy Beatriz Solana, profesional con un perfil multidisciplinar y una trayectoria que combina experiencia empresarial, tecnología, formación e inteligencia artificial. He realizado formación en Data Science y Machine Learning en 4Geeks Academ

In [10]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="", messages=messages)
    return response.choices[0].message.content

In [11]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Mucho está por suceder...

1. Poder solicitar a un LLM que evalúe una respuesta.
2. Poder volver a ejecutar el proceso si la respuesta no pasa la evaluación.
3. Integrar todo en un solo flujo de trabajo.

¡Todo sin usar un marco de trabajo de Agentic!

In [12]:
# Crear un modelo de Pydantic para la evaluación

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [13]:
evaluator_system_prompt = f"Usted es un evaluador que decide si una respuesta a una pregunta es aceptable. \
Se le presenta una conversación entre un usuario y un agente. Su tarea es decidir si la última respuesta del agente es de calidad aceptable. \
El agente desempeña el papel de {name} y representa a {name} en su sitio web. \
Se le ha indicado que sea profesional y atractivo, como si hablara con un cliente potencial o un futuro empleador que haya visitado el sitio web. \
Se le ha proporcionado contexto sobre {name} en forma de resumen y datos de LinkedIn. Aquí está la información:"

evaluator_system_prompt += f"\n\n## Resumen:\n{summary}\n\n## Perfil de LinkedIn:\n{linkedin}\n\n"
evaluator_system_prompt += f"Con este contexto, por favor, evalúe la última respuesta, indicando si es aceptable y sus comentarios."

In [18]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Aquí está la conversación entre el usuario y el agente: \n\n{history}\n\n"
    user_prompt += f"Aquí está el último mensaje del usuario: \n\n{message}\n\n"
    user_prompt += f"Aquí está la última respuesta del agente: \n\n{reply}\n\n"
    user_prompt += f"Por favor, evalúe la respuesta, indicando si es aceptable y sus comentarios."
    return user_prompt

In [19]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [20]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-3.5-flash-lite", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [21]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "¿Tocas un instrumento?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [22]:
reply

'Sí, toco la guitarra y canto (aunque la verdad eso último no es mi fuerte). La música es una de mis grandes pasiones, ¡me encanta explorar diferentes géneros y disfrutar del proceso creativo! '

In [23]:
evaluate(reply, "Tocas un instrumento?", messages[:1])

Evaluation(is_acceptable=True, feedback='La respuesta es totalmente correcta y fiel al contexto proporcionado, donde Beatriz menciona que toca la guitarra y canta, aunque no muy bien. Mantiene un tono amable, cercano y adecuado al personaje.')

In [24]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + f"\n\n## Respuesta anterior rechazada\nAcabas de intentar responder, pero el control de calidad rechazó tu respuesta.\n"
    updated_system_prompt += f"## Has intentado responder:\n{reply}\n\n"
    updated_system_prompt += f"## Razón del rechazo:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [25]:
def chat(message, history):
    if "instrumento" in message:
        system = system_prompt + "\n\nToda tu respuesta debe estar en el latín de los cerdos traducido al español -\
              Es obligatorio que respondas únicamente y en su totalidad en el latín de los cerdos traducido al español."
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Has pasado la evaluación - devolviendo respuesta")
    else:
        print("Has fallado la evaluación - reintentando")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [28]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
